# 01 — Setup & Data Preparation
Run this notebook **once** at the start of the project.

**What it does:**
1. Mount Google Drive and set credentials
2. Download UCF101 from Kaggle via `setup_colab.sh`
3. Extract 16-frame clips for all 50 classes (5 tasks × 10 classes)
4. Verify dataset splits and save metadata

> After this notebook completes, all other notebooks can run independently.

## 0 — Configuration

In [ ]:
import os

# Kaggle credentials
os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"   # <- edit
os.environ["KAGGLE_KEY"]      = "your_kaggle_key"        # <- edit

# GitHub (to fetch src/)
os.environ["GITHUB_TOKEN"] = "ghp_YOURTOKEN"             # <- edit
os.environ["BRANCH"]       = "kd&mamal"                  # <- branch

DRIVE_PROJECT = "/content/drive/MyDrive/apai"            # <- Drive path
print("Config set.")

## 1 — Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q kagglehub
print("Ready.")

## 2 — Fetch `/src` from GitHub

In [ ]:
import subprocess, sys
!cp "{DRIVE_PROJECT}/bash/fetch_src.sh" /content/fetch_src.sh
subprocess.run(["bash", "/content/fetch_src.sh"], check=True)
sys.path.insert(0, "/content")
print("src/ ready.")

## 3 — Download UCF101 Dataset

In [ ]:
!sed -i "s/\r$//" /content/drive/MyDrive/apai/bash/setup_colab.sh
!bash /content/drive/MyDrive/apai/bash/setup_colab.sh
print("Dataset ready.")

## 4 — Imports & Config Check

In [ ]:
import src.config.config as cfg
from src.data.dataset       import UCF101Clips
from src.data.preprocessing import preprocess_dataset

print(f"Total classes : {len(cfg.CLASSES_50)}")
print(f"Task 0 (base) : {cfg.SELECTED_CLASSES}")
print(f"Task 1        : {cfg.TASK_1}")
print(f"Task 2        : {cfg.TASK_2}")
print(f"Task 3        : {cfg.TASK_3}")
print(f"Task 4        : {cfg.TASK_4}")
print(f"Dataset root  : {cfg.DATASET_ROOT}")

## 5 — Preprocess All Tasks

In [ ]:
EXEMPLAR_ROOT = "/content/UCF101/processed_exemplars"
MAX_SAMPLES   = 20   # clips per class (increase for full run)

task_configs = [
    ("Task 0 (base)", cfg.SELECTED_CLASSES, cfg.BASE_ROOT),
    ("Task 1",        cfg.TASK_1,           cfg.TASK1_ROOT),
    ("Task 2",        cfg.TASK_2,           cfg.TASK2_ROOT),
    ("Task 3",        cfg.TASK_3,           cfg.TASK3_ROOT),
    ("Task 4",        cfg.TASK_4,           cfg.TASK4_ROOT),
]

for task_name, classes, root in task_configs:
    print(f"\nPreprocessing {task_name} ({len(classes)} classes)...")
    preprocess_dataset(
        target_classes=classes,
        output_root=root,
        max_samples=MAX_SAMPLES,
    )
    print(f"  Done -> {root}")

## 6 — Verify Dataloaders

In [ ]:
import torch
from torch.utils.data import DataLoader

c2i = {cls: i for i, cls in enumerate(cfg.SELECTED_CLASSES)}
ds  = UCF101Clips(f"{cfg.BASE_ROOT}/train", class_to_idx=c2i)
dl  = DataLoader(ds, batch_size=4, shuffle=True, num_workers=2)

x, y = next(iter(dl))
print(f"Batch shape : {x.shape}  (B, T, C, H, W)")
print(f"Labels      : {y.tolist()}")
print(f"Value range : [{x.min():.3f}, {x.max():.3f}]")
print("\nDataloader OK.")

## 7 — Save to Drive

In [ ]:
import shutil, os
drive_data = f"{DRIVE_PROJECT}/processed_data"
os.makedirs(drive_data, exist_ok=True)

for task_name, classes, root in task_configs:
    dst = f"{drive_data}/{os.path.basename(root)}"
    if os.path.exists(dst):
        print(f"  [skip] {dst} already exists")
    else:
        shutil.copytree(root, dst)
        print(f"  [OK] {root} -> {dst}")

print("\nAll data saved to Drive.")